### Conditional Order (Makespan)
For makespan minimisation, suppose two separation-identical aircraft $i$ and $j$ satisfy $r_i \leq r_j$ and $lt_i \leq lt_j$. If the following inequality holds, then placing $i$ before $j$ yields a makespan no worse than the swapped sequence.

$$
W_1 (t_i - b_i)^{\alpha} + W_2 C(t_i, lc_i)
+ W_1 (t_j - b_j)^{\alpha} + W_2 C(t_j, lc_j)
\le\ W_1 (t_i^\prime - b_i)^{\alpha} + W_2 C(t_i^\prime, lc_i)
+ W_1 (t_j^\prime - b_j)^{\alpha} + W_2 C(t_j^\prime, lc_j)
$$


In [2]:
from cvc5 import Kind
from core.context import *
from core.checks import *

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()
solver = ctx.solver

# Left and right sides of the new two-aircraft inequality.
def C(t, lc):
    return solver.mkTerm(
        Kind.ITE,
        solver.mkTerm(Kind.LEQ, t, lc),
        solver.mkReal("0"),
        solver.mkTerm(
            Kind.ITE,
            solver.mkTerm(Kind.LEQ, t, solver.mkTerm(Kind.ADD, lc, solver.mkReal("300"))),
            solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, t, lc), solver.mkReal("2")),
            solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.MULT, solver.mkReal("3"), solver.mkTerm(Kind.SUB, t, lc)), solver.mkReal("4")),
        ),
    )

lhs = solver.mkTerm(
    Kind.ADD,
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S1.takeoff["i"], ctx.b["i"]), C(S1.takeoff["i"], ctx.lc["i"])),
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S1.takeoff["j"], ctx.b["j"]), C(S1.takeoff["j"], ctx.lc["j"])),
)
rhs = solver.mkTerm(
    Kind.ADD,
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S2.takeoff["i"], ctx.b["i"]), C(S2.takeoff["i"], ctx.lc["i"])),
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S2.takeoff["j"], ctx.b["j"]), C(S2.takeoff["j"], ctx.lc["j"])),
)

# Encode and enforce the rule premises; total order on release/tw times and that i and j are delta-equivalent.
premises = [
    solver.mkTerm(Kind.LEQ, ctx.r["i"], ctx.r["j"]),  solver.mkTerm(Kind.LEQ, ctx.lt["i"], ctx.lt["j"]),
    solver.mkTerm(Kind.LEQ, ctx.b["i"], ctx.b["j"]),  solver.mkTerm(Kind.LEQ, ctx.lc["i"], ctx.lc["j"]),
    *ctx.separation_equivalence("i", "j"),
    solver.mkTerm(Kind.LEQ, lhs, rhs),
]

# Encode the claim; no greater makespan in S2 than in S1.
claim = solver.mkTerm(Kind.LEQ, S1.makespan, S2.makespan)

res = verify_pruning_rule(ctx, premises, claim)
print(f"Conditional Order Makespan (total):           {res}")


Conditional Order Makespan (total):           Verified


### Conditional Order (Delay & CTOT)
Using the same release-time, latest-time, and separation-equivalence premises, if the following inequality holds, then the combined delay-and-CTOT dominance check already in this notebook follows.

$$
W_1 (t_i - b_i)^{\alpha} + W_2 C(t_i, lc_i)
+ W_1 (t_j - b_j)^{\alpha} + W_2 C(t_j, lc_j)
\le\ W_1 (t_i^\prime - b_i)^{\alpha} + W_2 C(t_i^\prime, lc_i)
+ W_1 (t_j^\prime - b_j)^{\alpha} + W_2 C(t_j^\prime, lc_j)
$$

In [ ]:
from cvc5 import Kind
from core.context import *
from core.checks import *

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()
solver = ctx.solver

# Left and right sides of the new two-aircraft inequality.
def C(t, lc):
    return solver.mkTerm(
        Kind.ITE,
        solver.mkTerm(Kind.LEQ, t, lc),
        solver.mkReal("0"),
        solver.mkTerm(
            Kind.ITE,
            solver.mkTerm(Kind.LEQ, t, solver.mkTerm(Kind.ADD, lc, solver.mkReal("300"))),
            solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, t, lc), solver.mkReal("2")),
            solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.MULT, solver.mkReal("3"), solver.mkTerm(Kind.SUB, t, lc)), solver.mkReal("4")),
        ),
    )

lhs = solver.mkTerm(
    Kind.ADD,
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S1.takeoff["i"], ctx.b["i"]), C(S1.takeoff["i"], ctx.lc["i"])),
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S1.takeoff["j"], ctx.b["j"]), C(S1.takeoff["j"], ctx.lc["j"])),
)
rhs = solver.mkTerm(
    Kind.ADD,
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S2.takeoff["i"], ctx.b["i"]), C(S2.takeoff["i"], ctx.lc["i"])),
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S2.takeoff["j"], ctx.b["j"]), C(S2.takeoff["j"], ctx.lc["j"])),
)

# Encode and enforce the rule premises; total order on release/tw times and that i and j are delta-equivalent.
premises = [
    solver.mkTerm(Kind.LEQ, ctx.r["i"], ctx.r["j"]),  solver.mkTerm(Kind.LEQ, ctx.lt["i"], ctx.lt["j"]),
    solver.mkTerm(Kind.LEQ, ctx.b["i"], ctx.b["j"]),  solver.mkTerm(Kind.LEQ, ctx.lc["i"], ctx.lc["j"]),
    *ctx.separation_equivalence("i", "j"),
    solver.mkTerm(Kind.LEQ, lhs, rhs),
]

# Encode the claim; no aircraft is more delayed in S1 than S2 (aside i and j).
claim = solver.mkTerm(Kind.AND, *[
    solver.mkTerm(
        Kind.LEQ,
        solver.mkTerm(Kind.ADD, S1.ctot[x], S1.delay[x]),
        solver.mkTerm(Kind.ADD, S2.ctot[x], S2.delay[x]),
    )
    for x in ctx.aircraft if x not in ("i", "j")
])

res = verify_pruning_rule(ctx, premises, claim)
print(f"\nConditional Order Delay & CTOT (per not i/j ac):  {res}\n")

# Encode the claim; total (delay + ctot) in S1 <= total (delay + ctot) in S2.
claim = solver.mkTerm(
    Kind.LEQ,
    solver.mkTerm(Kind.ADD, *[solver.mkTerm(Kind.ADD, S1.ctot[x], S1.delay[x]) for x in ctx.aircraft]),
    solver.mkTerm(Kind.ADD, *[solver.mkTerm(Kind.ADD, S2.ctot[x], S2.delay[x]) for x in ctx.aircraft]),
)

res = verify_pruning_rule(ctx, premises, claim)
print(f"\nConditional Order Delay & CTOT (total):           {res}\n")


### Conditional Order (Delay)
For delay-based objectives, under $r_i \leq r_j$, $lt_i \leq lt_j$, and separation equivalence, if the following inequality holds, then placing $i$ before $j$ yields a schedule whose delay is no worse than the swapped sequence, both per aircraft and in total.

$$
W_1 (t_i - b_i)^{\alpha} + W_2 C(t_i, lc_i)
+ W_1 (t_j - b_j)^{\alpha} + W_2 C(t_j, lc_j)
\le\ W_1 (t_i^\prime - b_i)^{\alpha} + W_2 C(t_i^\prime, lc_i)
+ W_1 (t_j^\prime - b_j)^{\alpha} + W_2 C(t_j^\prime, lc_j)
$$


In [ ]:
from cvc5 import Kind
from core.context import *
from core.checks import *

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()
solver = ctx.solver

# Left and right sides of the new two-aircraft inequality.
def C(t, lc):
    return solver.mkTerm(
        Kind.ITE,
        solver.mkTerm(Kind.LEQ, t, lc),
        solver.mkReal("0"),
        solver.mkTerm(
            Kind.ITE,
            solver.mkTerm(Kind.LEQ, t, solver.mkTerm(Kind.ADD, lc, solver.mkReal("300"))),
            solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, t, lc), solver.mkReal("2")),
            solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.MULT, solver.mkReal("3"), solver.mkTerm(Kind.SUB, t, lc)), solver.mkReal("4")),
        ),
    )

lhs = solver.mkTerm(
    Kind.ADD,
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S1.takeoff["i"], ctx.b["i"]), C(S1.takeoff["i"], ctx.lc["i"])),
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S1.takeoff["j"], ctx.b["j"]), C(S1.takeoff["j"], ctx.lc["j"])),
)
rhs = solver.mkTerm(
    Kind.ADD,
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S2.takeoff["i"], ctx.b["i"]), C(S2.takeoff["i"], ctx.lc["i"])),
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S2.takeoff["j"], ctx.b["j"]), C(S2.takeoff["j"], ctx.lc["j"])),
)

# Encode and enforce the rule premises; total order on release/tw times and that i and j are delta-equivalent.
premises = [
    solver.mkTerm(Kind.LEQ, ctx.r["i"], ctx.r["j"]),  solver.mkTerm(Kind.LEQ, ctx.lt["i"], ctx.lt["j"]),
    solver.mkTerm(Kind.LEQ, ctx.b["i"], ctx.b["j"]),  solver.mkTerm(Kind.LEQ, ctx.lc["i"], ctx.lc["j"]),
    *ctx.separation_equivalence("i", "j"),
    solver.mkTerm(Kind.LEQ, lhs, rhs),
]

# Verify that every aircraft other than i and j incurs no larger delay in S1.
claim = solver.mkTerm(Kind.AND, *[
    solver.mkTerm(Kind.LEQ, S1.delay[x], S2.delay[x])
    for x in ctx.aircraft if x not in ("i", "j")
])

res = verify_pruning_rule(ctx, premises, claim)
print(f"\nConditional Order Delay (per-aircraft): {res}\n")

# Verify that total delay in S1 is no larger than in S2.
claim = solver.mkTerm(
    Kind.LEQ,
    solver.mkTerm(Kind.ADD, *[S1.delay[x] for x in ctx.aircraft]),
    solver.mkTerm(Kind.ADD, *[S2.delay[x] for x in ctx.aircraft]),
)

res = verify_pruning_rule(ctx, premises, claim)
print(f"\nConditional Order Delay (total):        {res}\n")


### Conditional Order (Time Window Feasibility)
For time-window feasibility, under the same release-time, latest-time, and separation-equivalence premises, if the following inequality holds and the swapped sequence is feasible, then placing $i$ before $j$ does not introduce any new time-window violation.

$$
W_1 (t_i - b_i)^{\alpha} + W_2 C(t_i, lc_i)
+ W_1 (t_j - b_j)^{\alpha} + W_2 C(t_j, lc_j)
\le\ W_1 (t_i^\prime - b_i)^{\alpha} + W_2 C(t_i^\prime, lc_i)
+ W_1 (t_j^\prime - b_j)^{\alpha} + W_2 C(t_j^\prime, lc_j)
$$

In [ ]:
from cvc5 import Kind
from core.context import *
from core.checks import *

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()
solver = ctx.solver

# Left and right sides of the new two-aircraft inequality.
def C(t, lc):
    return solver.mkTerm(
        Kind.ITE,
        solver.mkTerm(Kind.LEQ, t, lc),
        solver.mkReal("0"),
        solver.mkTerm(
            Kind.ITE,
            solver.mkTerm(Kind.LEQ, t, solver.mkTerm(Kind.ADD, lc, solver.mkReal("300"))),
            solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, t, lc), solver.mkReal("2")),
            solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.MULT, solver.mkReal("3"), solver.mkTerm(Kind.SUB, t, lc)), solver.mkReal("4")),
        ),
    )

lhs = solver.mkTerm(
    Kind.ADD,
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S1.takeoff["i"], ctx.b["i"]), C(S1.takeoff["i"], ctx.lc["i"])),
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S1.takeoff["j"], ctx.b["j"]), C(S1.takeoff["j"], ctx.lc["j"])),
)
rhs = solver.mkTerm(
    Kind.ADD,
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S2.takeoff["i"], ctx.b["i"]), C(S2.takeoff["i"], ctx.lc["i"])),
    solver.mkTerm(Kind.ADD, solver.mkTerm(Kind.SUB, S2.takeoff["j"], ctx.b["j"]), C(S2.takeoff["j"], ctx.lc["j"])),
)

# Encode and enforce the rule premises; total order on release/tw times and that i and j are delta-equivalent.
premises = [
    solver.mkTerm(Kind.LEQ, ctx.r["i"], ctx.r["j"]),  solver.mkTerm(Kind.LEQ, ctx.lt["i"], ctx.lt["j"]),
    solver.mkTerm(Kind.LEQ, ctx.b["i"], ctx.b["j"]),  solver.mkTerm(Kind.LEQ, ctx.lc["i"], ctx.lc["j"]),
    *ctx.separation_equivalence("i", "j"),
    solver.mkTerm(Kind.LEQ, lhs, rhs),
    *S2.time_window_feasible,
]

# Verify that S1 does not introduce any time-window violation absent from S2.
claim = solver.mkTerm(Kind.AND, *[
    solver.mkTerm(Kind.OR, solver.mkTerm(Kind.NOT, S2.window_violation[x]), S1.window_violation[x])
    for x in ctx.aircraft
])

res = verify_pruning_rule(ctx, premises, claim)
print(f"\nConditional Order Time Windows: {res}\n")
